# Chapter 2 - Lab 2: Portfolio Rebalancing Agent with Working Memory

## Notebook Overview

This lab takes the tool-use foundation one step further by adding **working memory**: it builds a portfolio-rebalancing agent that, rather than answering one isolated question, carries a full multi-turn conversation in its head — and the notebook shows, after every single turn, exactly what that memory contains. It also takes a first pass at compressing the conversation into a reusable "episode," a theme that Lab 3 develops fully. Along the way you see the four types of AI memory — working, episodic, semantic and procedural — all mapped onto this one finance scenario.

After installing `openai-agents` (with `langchain_openai` and `chromadb` coming later) and loading `OPENAI_API_KEY` from the root `.env` file, the notebook defines the **portfolio agent**: a bare `gpt-4o-mini` `Agent` with no tools, whose instructions tell it to analyze the current portfolio allocation against the market regime (volatility and trend), make assumptions explicit, and propose specific rebalancing trades with percentages and justifications.

The core of the lab is the **working memory setup**. The agent is given a session via the SDK's `SQLiteSession("conversation_ptf")`. This session object is the container that holds the whole conversation, and the lab explains its mechanics in detail: the session is just an identifier, while the actual data lives in a SQLite database with two tables — `agent_sessions` (keyed by `session_id`) and `agent_messages` (a row per message, storing each message as a JSON blob). Every time you call `Runner.run(agent, input=..., session=session_ptf_rebal)` the SDK appends your prompt and the agent's reply as new rows; nothing is ever overwritten. `get_items()` then replays those rows in insertion order and decodes them back into user/assistant message objects. That is why the memory appears to "grow": it is an append-only log keyed to the session. Because `SQLiteSession` defaults to an in-memory database, this history survives across cells in the running kernel but is wiped on restart unless you pass a file path.

With that in place, the notebook runs a **six-turn simulated conversation** about rebalancing. Query 1 presents a high-volatility, declining market with an initial allocation of 70% stocks / 20% bonds / 10% cash; query 2 has the user choosing 50/35/15; query 3 (high vol, uncertain trend) has the agent rebalance the new mix; query 4 moves to 45/40/15; query 5 shifts to a low-volatility, bullish regime; query 6 lands on 70/25/05. After most queries a "Display Working Memory" cell prints `get_items()`, so you watch the stored history accumulate from two items, to four, and so on up to twelve. The agent consistently reasons from the *latest* allocation because it remembers every prior turn — the exact behavior that disappears if you pass no session, or a fresh one. A "play with the agent yourself" cell and a short explanation of why working memory matters round out this section.

The second half starts the move to long-term memory under the heading **Generate Episodic Memory**. Using LangChain, the lab builds an episodic-memory prompt (`ChatPromptTemplate`) that instructs a model to read the raw conversation and compress it into structured JSON: `conversation_summary`, `portfolio_allocation_trajectory`, `what_worked`, `what_didnt_work` and `key_points`, with good and bad few-shot examples to guide the output. The conversation is flattened into a single string first, then fed to a `ChatOpenAI(temperature=0.7, model="gpt-4o-mini")` chained through `JsonOutputParser`, producing a structured episode. The same template is then re-sent through the OpenAI **Responses API** to show a second, equally valid way to generate the same episode. This lab also installs ChromaDB and embeds that episode with `text-embedding-3-small` into a persistent vector collection, then demonstrates retrieval: a brand-new query finds the similar past episode via `collection.query(...)`, which is appended to a prompt given to a fresh session (`conversation_ptf_2`) — the first glimpse of a lesson from a *past* conversation informing a *new* one.

A final **All in** section wraps the whole pipeline into reusable functions (`concat_working_memory`, `get_episodic_memory`, `add_data_to_episodic_memory`, `get_similar_episodes`) and runs an interactive `while True` loop under a new session: each `User:` prompt is augmented with retrieved similar episodes before going to the agent, and typing `exit` summarizes the session and stores the episode in ChromaDB under id `memory_2`. An empty-input guard prevents a blank line from being embedded and crashing the API call. Closing inspection cells show the stored document and its embeddings.

The two conceptual sections — **Semantic Memory** and **Procedural Memory** — contain no code. Semantic memory is the "same trick, different content": retrieve facts and concepts from a knowledge base and append them to the context, just as episodes were appended. Procedural memory is the "how-to" counterpart: for a portfolio-rebalancing workflow, that would be the stored recipe of identifying the current allocation, comparing it to the target, and proposing trades.

**The point of the lab:** working memory is what keeps a conversation coherent during a session. Its weakness is that it *ends with the session* — which is exactly the gap the next lab closes by turning the conversation into durable, retrievable episodes.



In this tutorial, we'll use an agent responsible for generating clear and explainable portfolio rebalancing recommendations.

We will build a discussion with the agent: **Conversation History**. This will constitute our **working memory**.


# Install libs

In [ ]:
# !pip install openai-agents -q

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd().parent / ".env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY


In [ ]:
from agents import Agent, Runner, function_tool, SQLiteSession
import nest_asyncio
nest_asyncio.apply()

# Rebalancing Portfolio Agent

Create portfolio rebalancing agent.

In this agent, I didn't add tools to interact with or any other reflective mechanisms. It's a simple agent that I'll be using to build a memory through the conversation.


In [ ]:
portfolio_agent = Agent(
    name="Portfolio Rebalancing Agent",
    instructions="""You are a Portfolio Rebalancing Agent for a professional investor.
Your goal is to propose clear, explainable rebalancing actions that keep the portfolio aligned with the target risk/return profile and constraints.

Your role
- Analyze the current portfolio (weights, asset classes, sectors, factors, cash).
- Incorporate market regime information (volatility, trends, macro environment).
- Compare current allocation to:
    - Target allocation or risk budget (if provided),
    - Or to a reasonable diversified allocation based on the information you have.
- Propose specific rebalancing trades with approximate percentage changes and short rationales.
- Prioritize diversification, risk control, and alignment with the investor’s objectives.
- Keep answers concise and structured, and state assumptions when information is missing.
- Treat the allocation provided in my prompot as the portfolio's current allocation.

Always explain your reasoning.""",
    model="gpt-4o-mini"
)


You can of course imagine an agent using tools connecting to:
* Your database (or other sources), to extract the current situation of your portfolio.
* Market data to compute volatility, trend....

This way, your agent can analyze your current portfolio allocation and recommend appropriate rebalancing actions using real market data.

# Set up Working Memory

Let’s instantiate the working memory, referred to as a session in the OpenAI Agent SDK:

In [ ]:
# Create a session instance that will persist across runs
session_id = "conversation_ptf"
session_ptf_rebal = SQLiteSession(session_id)

The SQLiteSession class provides a session mechanism that stores the interaction history in a SQLite database, enabling persistence between runs. The variable session_id uniquely names the session and allows the system to associate stored messages and state with this specific conversation.

## User Query 1

We'll add the session memory parameter to the agent's run. This ensures that the entire interaction is stored and maintained as part of the working memory.

We provide the agent with the current market context, such as the volatility regime and market trend, along with the current portfolio allocation, and ask it to recommend appropriate rebalancing actions.


In [ ]:
input_user_1 = """Current Market Situation:
- Volatility Regime: high
- Market Trend: declining

Current Portfolio State:
- Portfolio: {"stocks": 0.70, "bonds": 0.20, "cash": 0.10}

What rebalancing action should I take?"""

output_1 = await Runner.run(
    starting_agent=portfolio_agent,
    input=input_user_1,
    session=session_ptf_rebal
)

print(f"Agent Decision:\n{output_1.final_output}\n")

The agent begins by outlining the current portfolio allocation, then states its assumptions and presents a recommended target allocation by asset class. It also provides a detailed rationale for each proposed adjustment:
* Reduce stock allocation from 70% to 50%
* Increase bond allocation from 20% to 40%
* Maintain cash at 10%


### Display Working Memory

Here we display the working memory at this point in the interaction with the agent:

In [ ]:
all_items = await session_ptf_rebal.get_items()

conversation_history = []
for i, msg in enumerate(all_items, 1):
    role = msg.get("role", "unknown")
    content = msg.get("content", "")
    if role == "assistant" :
      print(f"  {i}. \033[95m{role}\033[0m: {content[0]['text']}")
      conversation_history.append((role, content[0]['text']))
      print("--"*50)
    if role == "user":
      print(f"  {i}. \033[95m{role}\033[0m: {content}")
      conversation_history.append((role, content))
      print("--"*50)

## User Query 2

We now ask a follow-up question while keeping the session parameter during execution, ensuring that the conversation history is preserved:

In [ ]:
input_user_2="In this case, I'm reducing equity to 50%, increasing bonds to 35% and cash to 15%."

output_2 = await Runner.run(
        starting_agent=portfolio_agent,
        input=input_user_2,
        session=session_ptf_rebal
    )

print(f"{output_2.final_output}\n")

### Display Working Memory

We now display the  agent’s working memory following the execution of the  two previous queries:

In [ ]:
all_items = await session_ptf_rebal.get_items()

conversation_history = []
for i, msg in enumerate(all_items, 1):
    role = msg.get("role", "unknown")
    content = msg.get("content", "")
    if role == "assistant" :
      print(f"  {i}. \033[95m{role}\033[0m: {content[0]['text']}")
      conversation_history.append((role, content[0]['text']))
      print("--"*50)
    if role == "user":
      print(f"  {i}. \033[95m{role}\033[0m: {content}")
      conversation_history.append((role, content))
      print("--"*50)

As you can see, all the discussion is stored in the working memory (session).

## User Query 3

Here is another request, still with the session parameter specified during the run:

In [ ]:
input_user_3 = """Current Market Situation:
- Volatility Regime: high
- Market Trend: uncertain

What rebalancing action should I take?"""

output_3 = await Runner.run(
        starting_agent=portfolio_agent,
        input=input_user_3,
        session=session_ptf_rebal
    )

print(f"{output_3.final_output}\n")

### Display Working Memory

In [ ]:
all_items = await session_ptf_rebal.get_items()

conversation_history = []
for i, msg in enumerate(all_items, 1):
    role = msg.get("role", "unknown")
    content = msg.get("content", "")
    if role == "assistant" :
      print(f"  {i}. \033[95m{role}\033[0m: {content[0]['text']}")
      conversation_history.append((role, content[0]['text']))
      print("--"*50)
    if role == "user":
      print(f"  {i}. \033[95m{role}\033[0m: {content}")
      conversation_history.append((role, content))
      print("--"*50)

## User Query 4

In [ ]:
input_user_4 = """In this case, I reduce equity to 45%, increase bonds to 40% and keep cash unchanged"""

output_4 = await Runner.run(
        starting_agent=portfolio_agent,
        input=input_user_4,
        session=session_ptf_rebal
    )

print(f"{output_4.final_output}\n")

### Display Working Memory

In [ ]:
all_items = await session_ptf_rebal.get_items()

conversation_history = []
for i, msg in enumerate(all_items, 1):
    role = msg.get("role", "unknown")
    content = msg.get("content", "")
    if role == "assistant" :
      print(f"  {i}. \033[95m{role}\033[0m: {content[0]['text']}")
      conversation_history.append((role, content[0]['text']))
      print("--"*50)
    if role == "user":
      print(f"  {i}. \033[95m{role}\033[0m: {content}")
      conversation_history.append((role, content))
      print("--"*50)

## User Query 5

In [ ]:
input_user_5 = """Current Market Situation:
- Volatility Regime: low
- Market Trend: bullish

What rebalancing action should I take?"""

output_5 = await Runner.run(
        starting_agent=portfolio_agent,
        input=input_user_5,
        session=session_ptf_rebal
    )

print(f"{output_5.final_output}\n")

## User Query 6

In [ ]:
input_user_6 = """in this situation, I'll increase stock to 70%, reduce bonds to 25% and cash to 5%"""

output_6 = await Runner.run(
        starting_agent=portfolio_agent,
        input=input_user_6,
        session=session_ptf_rebal
    )

print(f"{output_6.final_output}\n")

### Display Final Working Memory

In [ ]:
all_items = await session_ptf_rebal.get_items()

conversation_history = []
for i, msg in enumerate(all_items, 1):
    role = msg.get("role", "unknown")
    content = msg.get("content", "")
    if role == "assistant" :
      print(f"  {i}. \033[95m{role}\033[0m: {content[0]['text']}")
      conversation_history.append((role, content[0]['text']))
      print("--"*50)
    if role == "user":
      print(f"  {i}. \033[95m{role}\033[0m: {content}")
      conversation_history.append((role, content))
      print("--"*50)

In [ ]:
# conversation_history_string = "/n".join([f"{elem[0]}: {elem[1]}" for elem in conversation_history])

# conversation_history_string+='user: p'

You can play with the agent yourself:

In [ ]:
# input_user_7 = """ Write here your prompt
# """

# output_7 = await Runner.run(
#         starting_agent=portfolio_agent,
#         input=input_user_7,
#         session=session_ptf_rebal
#     )

# print(f"{output_7.final_output}\n")

Working memory is essential in this context.

Without it, the agent would not retain the initial portfolio allocation, {"stocks": 0.70, "bonds": 0.20, "cash": 0.10}, and would therefore be unable to correctly apply subsequent instructions, such as increasing bonds to 25%, and reducing cash to 5%.

Without access to the initial state, the agent would not compute or recommend an appropriate rebalancing strategy.

# Generate Episodic Memory

## LangChain install

In [ ]:
# !pip install langchain_openai -q

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(temperature=0.7, model="gpt-4o-mini")

### Prompt of episodic memory: LangChain prompt template

In [ ]:
# from langchain_core.prompts import ChatPromptTemplate
# from langchain_core.output_parsers import JsonOutputParser

episodic_memory_prompt_template = """You are an episodic memory agent supporting a principal portfolio-rebalancing agent.
Your job is to create a concise episodic memory of the session:
* Summarize the conversation (in conversation_summary),
* Store a structured trajectory of portfolio allocations and their corresponding market situations, capturing how allocations evolved over time and under which conditions (in portfolio_trajectory).
* Identify what worked (in what_worked) and what didn’t (in what_didnt_work),
* Extract key points (in key_points).

You receive the full conversation history between the user and the principal agent.
The portfolio includes equities, bonds, and cash, and the discussion covers the market situation, portfolio state and outcome, and rebalancing actions.

OUPUT FORMAT:
Return only valid JSON in this exact schema:
{{

	"conversation_summary": string, // 3–5 sentences summarizing: market conditions, portfolio situation, actions taken or discussed.
	"portfolio_allocation_trajectory": string, // Portfolio trajectory recording how portfolio allocations evolve across sessions together with the associated market situations.
	"what_worked": string, //Short analysis of what was effective: alignment with goals, good risk management, coherent rebalancing reasoning, improved diversification, clear decisions.
	"what_didnt_work": string, //Short analysis of frictions: unclear strategy, missed opportunities, over/under-reaction to market signals, concentration risk, unclear constraints, data issues
	"key_points": ["string"] // List of the most important insights or decisions to keep for future sessions.
}}


FEW-SHOT EXAMPLES
Here are some examples of what is good to produce and what is bad to avoid:

GOOD conversation_summary — Example
“Market uncertainty increased due to rising yields and mixed earnings. The portfolio was equity-heavy at 65%, creating higher volatility. The user discussed reducing equity risk and reallocating toward short-duration bonds. A partial rebalance was executed, bringing equities to 55% and increasing cash reserves. The agent emphasized maintaining flexibility given macro uncertainty.”

BAD conversation_summary — Example
“The market was bad. The portfolio changed. Some trades were done. It was about rebalancing.”
(Too vague, no numbers, no context, no actions.)

GOOD what_worked — Example
“The shift from long-duration bonds to shorter maturities improved interest-rate resilience. The reduction of tech overweight aligned well with the user’s lower risk tolerance. The conversation clearly linked macro signals to rebalancing decisions.”

BAD what_worked — Example
“Everything worked well.” (Empty, no link to portfolio or market.)
“The agent was nice.” (Irrelevant.)

GOOD what_didnt_work — Example:
“The user’s liquidity needs were mentioned but not fully integrated into the new allocation. Equity risk remains high despite concerns about earnings volatility. Market data gaps created hesitation about bond exposure.”

BAD what_didnt_work — Example:
“Nothing went wrong.” (Unrealistic; lacks analysis.)
“The user disagreed with the agent.” (Not informative, no portfolio implications.)

General Rules:
- Be concise and factual.
- Do not invent trades or market context not present in the conversation.
- Capture insights, not generic statements.
- Output only valid JSON and nothing else.

Here is the conversation between the user and the principal portfolio-rebalancing agent:
{conversation}
"""

episodic_memory_prompt = ChatPromptTemplate.from_template(episodic_memory_prompt_template)

generate_episodic_memory = episodic_memory_prompt | llm | JsonOutputParser()

In [ ]:
print(episodic_memory_prompt.messages[0].prompt.template)

### Conversation History

In [ ]:
all_items = await session_ptf_rebal.get_items()

conversation_history = []
for i, msg in enumerate(all_items, 1):
    role = msg.get("role", "unknown")
    content = msg.get("content", "")
    if role == "assistant" :
      print(f"  {i}. \033[95m{role}\033[0m: {content[0]['text']}")
      conversation_history.append((role, content[0]['text']))
      print("--"*50)
    if role == "user":
      print(f"  {i}. \033[95m{role}\033[0m: {content}")
      conversation_history.append((role, content))
      print("--"*50)

### Concat conversation history

In [ ]:
conversation_history_string = "\n\n".join([f'{elem[0]}: """{elem[1]}"""' for elem in conversation_history])

print(conversation_history_string)

### Generate Episodic Memory with LangChain

In [ ]:
episodic_memory = generate_episodic_memory.invoke({"conversation": conversation_history_string})
episodic_memory

In [ ]:
# You can also add the whole conversation history to the episodic memory.
# Or add only the last 4 messages (user & assistant)
# episodic_memory
# str_json = f"""{'conversation_history': {conversation_history_string},
#   'conversation_summary': 'The market shifted to a low volatility and bullish trend. The user initially held a portfolio of 45% stocks, 40% bonds, and 15% cash. They decided to significantly increase equity exposure to 70%, reduce bond allocation to 25%, and decrease cash reserves to 5%. This restructuring aims to capitalize on the positive market conditions while accepting higher risk.',
#  'portfolio_allocation_trajectory': 'Session 1: Stocks 70%, Bonds 20%, Cash 10% (High volatility, declining market); Session 2: Stocks 50%, Bonds 35%, Cash 15% (High volatility, declining market); Session 3: Stocks 45%, Bonds 40%, Cash 15% (High volatility, uncertain market); Session 4: Stocks 55%, Bonds 30%, Cash 15% (Low volatility, bullish market); Session 5: Stocks 70%, Bonds 25%, Cash 5% (Low volatility, bullish market)',
#  'what_worked': 'The decision to significantly increase stock allocation aligns well with the bullish market trend, leveraging potential for higher returns. The strategy demonstrates effective risk management by reducing bonds and cash in favor of equities, optimizing the portfolio for growth.',
#  'what_didnt_work': 'The substantial reduction in bonds and cash may expose the portfolio to greater risk, particularly if market conditions fluctuate unexpectedly. A lower cash reserve could limit liquidity for future opportunities or risk mitigation.',
#  'key_points': ['Leverage bullish market conditions by increasing equity exposure.',
#   'Maintain a balance between risk and potential returns when adjusting allocations.',
#   'Monitor market conditions closely after significant rebalancing actions.']}"""

In [ ]:
str_json = """{'conversation_summary': 'The market shifted to a low volatility and bullish trend. The user initially held a portfolio of 45% stocks, 40% bonds, and 15% cash. They decided to significantly increase equity exposure to 70%, reduce bond allocation to 25%, and decrease cash reserves to 5%. This restructuring aims to capitalize on the positive market conditions while accepting higher risk.',
 'portfolio_allocation_trajectory': 'Session 1: Stocks 70%, Bonds 20%, Cash 10% (High volatility, declining market); Session 2: Stocks 50%, Bonds 35%, Cash 15% (High volatility, declining market); Session 3: Stocks 45%, Bonds 40%, Cash 15% (High volatility, uncertain market); Session 4: Stocks 55%, Bonds 30%, Cash 15% (Low volatility, bullish market); Session 5: Stocks 70%, Bonds 25%, Cash 5% (Low volatility, bullish market)',
 'what_worked': 'The decision to significantly increase stock allocation aligns well with the bullish market trend, leveraging potential for higher returns. The strategy demonstrates effective risk management by reducing bonds and cash in favor of equities, optimizing the portfolio for growth.',
 'what_didnt_work': 'The substantial reduction in bonds and cash may expose the portfolio to greater risk, particularly if market conditions fluctuate unexpectedly. A lower cash reserve could limit liquidity for future opportunities or risk mitigation.',
 'key_points': ['Leverage bullish market conditions by increasing equity exposure.',
  'Maintain a balance between risk and potential returns when adjusting allocations.',
  'Monitor market conditions closely after significant rebalancing actions.']}"""

import json
str_json_fixed = str_json.replace("'", '"')
episodic_memory_json = json.loads(str_json_fixed)

In [ ]:
# str_json = """{'conversation_summary': 'The user initially outlined a portfolio with a high volatility regime and a declining market trend, proposing to reduce equity to 50%, increase bonds to 35%, and cash to 15%. The agent confirmed the proposed adjustments, rebalance actions were discussed, and an allocation totaling 100% was outlined. The new allocation was set to 50% stocks, 35% bonds, and 15% cash, focusing on risk management amidst market uncertainty.',
#  'what_worked': 'The conversation effectively aligned the user’s risk tolerance with rebalancing strategies, emphasizing a reduction in equities to manage volatility while increasing bonds for stability. The proposed changes maintain a structured approach to asset allocation.',
#  'what_didnt_work': 'Initially, the total asset allocation did not sum to 100%, which could lead to confusion. There was also a need for a clearer rationale behind the cash reduction to ensure liquidity was adequately addressed.',
#  'key_points': ['Maintain stock exposure at 50% to balance growth potential.',
#   'Increase bond exposure to 45% for enhanced stability.',
#   'Reduce cash allocation to 5% to ensure total allocation sums to 100%.']}"""

In [ ]:
# import ast
# episodic_memory_json = ast.literal_eval(str_json)
# episodic_memory

## Responses API from OpenAI

In [ ]:
user_prompt = episodic_memory_prompt_template.format(
    conversation=conversation_history_string
)
print(user_prompt)

In [ ]:
from openai import OpenAI
client = OpenAI(api_key = OPENAI_API_KEY)

response = client.responses.create(
  model="gpt-4.1-mini",
  input=[{"role":"user","content":user_prompt}],
  text = {"format":{"type": "json_object"}},
)

print(json.dumps(json.loads(response.output[0].content[0].text), indent=2))

# ChromaDB client

In [ ]:
!pip install chromadb -q

In [ ]:
import chromadb

from chromadb.utils import embedding_functions

# Define an OpenAI embedding function using the specified embedding model
embedding_function  = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY, #Your OpenAI API Key
    model_name="text-embedding-3-small",
)

# Name of the collection used to store episodic memory embeddings
collection_name="episodic_memory_ptf_rebal"

# Create a persistent ChromaDB client with on-disk storage
persistent_client = chromadb.PersistentClient(path="./chroma_ptf_rebal")

# Create or load the collection and associate it with the embedding function
collection = persistent_client.get_or_create_collection(collection_name, embedding_function=embedding_function )

In [ ]:
persistent_client.get_collection(name='episodic_memory_ptf_rebal').get_model()

## Add Memory in DB

In [ ]:
episodic_memory_json

In [ ]:
episodic_memory

In [ ]:

document_text = f"""
Conversation Summary: {episodic_memory['conversation_summary']}
Portfolio Allocation Trajectory: {episodic_memory['portfolio_allocation_trajectory']}
What Worked: {episodic_memory['what_worked']}
What Didn't Work: {episodic_memory['what_didnt_work']}
Key Points: {', '.join(episodic_memory['key_points'])}
"""

metadata_clean = {
    "conversation_summary": episodic_memory["conversation_summary"],
    "portfolio_allocation_trajectory": episodic_memory["portfolio_allocation_trajectory"],
    "what_worked": episodic_memory["what_worked"],
    "what_didnt_work": episodic_memory["what_didnt_work"],
    "key_points": " | ".join(episodic_memory["key_points"]), #Convert a list to a string to be added in chromDB
}

collection.add(
    ids=["memory_1"],  # unique id
    documents=[document_text],  # main text
    metadatas=[metadata_clean],  # full dict as metadata (JSON-serializable)
)


## Query Memory

In [ ]:
input_user_init = """Current Market Situation:
- Volatility Regime: low
- Market Trend: bullish

What rebalancing action should I take?"""

episodic_memory_recap = collection.query(
    query_texts=[input_user_init],
    n_results=3,
)
episodic_memory_recap

In [ ]:
# context_memory =
episodic_memory_recap['documents'][0]

## Call the agent

In [ ]:
#Create a new session that will persist across runs
session_id = "conversation_ptf_2"
session_ptf_rebal_2 = SQLiteSession(session_id)

In [ ]:
input_user_init = """Current Market Situation:
- Volatility Regime: low
- Market Trend: bullish

What rebalancing action should I take?"""

input_user_with_memory = input_user_init + f""""

Here are relevant insights from past rebalancing episodes. Use them as guidance, not strict rules:
{episodic_memory_recap['documents'][0][0]}

Argue your proposition using past rebalancing episodes if needed.
"""

output_using_memory = await Runner.run(
        starting_agent=portfolio_agent,
        input=input_user_with_memory,
        session=session_ptf_rebal_2
    )

print(f"{output_using_memory.final_output}\n")

In [ ]:
all_items = await session_ptf_rebal_2.get_items()

conversation_history_2 = []
for i, msg in enumerate(all_items, 1):
    role = msg.get("role", "unknown")
    content = msg.get("content", "")
    if role == "assistant" :
      print(f"  {i}. \033[95m{role}\033[0m: {content[0]['text']}")
      conversation_history_2.append((role, content[0]['text']))
      print("--"*50)
    if role == "user":
      print(f"  {i}. \033[95m{role}\033[0m: {content}")
      conversation_history_2.append((role, content))
      print("--"*50)

## All in

In [ ]:
def concat_working_memory(all_items):
  conversation_history = []
  for i, msg in enumerate(all_items, 1):
      role = msg.get("role", "unknown")
      content = msg.get("content", "")
      if role == "assistant" :
        conversation_history.append((role, content[0]['text']))
      if role == "user":
        conversation_history.append((role, content))
  return conversation_history

In [ ]:
def get_episodic_memory(conversation_history):
  conversation_history_string = "\n\n".join([f'{elem[0]}: """{elem[1]}"""' for elem in conversation_history])
  episodic_memory = generate_episodic_memory.invoke({"conversation": conversation_history_string})
  return episodic_memory

In [ ]:
def add_data_to_episodic_memory(episodic_memory, collection_id):
  document_text = f"""Conversation Summary: {episodic_memory['conversation_summary']}
  Portfolio Allocation Trajectory: {episodic_memory['portfolio_allocation_trajectory']}
  What Worked: {episodic_memory['what_worked']}
  What Didn't Work: {episodic_memory['what_didnt_work']}
  Key Points: {', '.join(episodic_memory['key_points'])}
  """

  metadata_clean = {
      "conversation_summary": episodic_memory["conversation_summary"],
      "portfolio_allocation_trajectory": episodic_memory["portfolio_allocation_trajectory"],
      "what_worked": episodic_memory["what_worked"],
      "what_didnt_work": episodic_memory["what_didnt_work"],
      "key_points": " | ".join(episodic_memory["key_points"]), #Convert a list to a string to be added in chromDB
  }

  collection.add(
      ids=[collection_id],  # unique id
      documents=[document_text],  # main text
      metadatas=[metadata_clean],  # full dict as metadata (JSON-serializable)
  )

In [ ]:
def get_similar_episodes(input_user_init):
  episodic_memory_recap = collection.query(
    query_texts=[input_user_init],
    n_results=3,
  )

  input_user_with_memory = input_user_init + f"""
    Here are relevant insights from past rebalancing episodes. Use them as guidance, not strict rules:
    {episodic_memory_recap['documents'][0][0]}

    Argue your propsition using past rebalacing episodes if needed.
    """
  return input_user_with_memory

In [ ]:
#Initiate Working Memory
session_id = "conversation_ptf_3"
session_ptf_rebal_3 = SQLiteSession(session_id)

#Define the id collection. Conversation will be stored in the episodic memory when the session ends
collection_id = 'memory_2'
all_items = []

while True:
    # Get User's Input
    input_user_init = input("\nUser: ")

    if input_user_init.strip() == "":
        print("Please type a message (or 'exit' to end).")
        continue

    if input_user_init.strip() == "exit":
        if all_items!=[]:
          print("End of the discussion, reformulating working memory to be added to episodic memory")
          conversation_history = concat_working_memory(all_items)
          episodic_memory = get_episodic_memory(conversation_history)
          print("---"*50)
          print("Episodic Memory to be stored: \n")
          print(episodic_memory)
          print("---"*50)
          add_data_to_episodic_memory(episodic_memory, collection_id)
        break

    #Extract episodic memory and build final prompt
    input_user_with_memory = get_similar_episodes(input_user_init)


    output_using_memory = await Runner.run(
            starting_agent=portfolio_agent,
            input=input_user_with_memory,
            session=session_ptf_rebal_3
        )

    print(f"{output_using_memory.final_output}\n")

    all_items = await session_ptf_rebal_3.get_items()


input_user_init = """Current Market Situation:
- Volatility Regime: low
- Market Trend: bullish

# What rebalancing action should I take?"""

# I will not change stock exposure, however I'll decrease the bonds to 20% and increase the cash to 10%

In [ ]:
all_items

In [ ]:
collection.get(ids=[collection_id])

In [ ]:
collection.get(
    ids=[collection_id],
    include=["embeddings"]
)


# Semantic Memory

Using the same logic (then in episodic memory) of appending the current context with new data, your can extract, using **RAG**, new information such as **facts** or **concepts** from a knowledge database.


This semantic memory is not about past experiences, but it's more about external knowledge your agent may need.

In porftolio rebalancing, this could mean adding **financial analysis** about **sectors** or **macro economic conditions**.

This kind of information can be stored in vector databases using RAG, allowing you to retrieve the most relevant insights when needed (related to the user query) and add them to the agent's context.

# Procedural Memory

A procedural memory is more "how to" perform a certain action.

For example, it could be, for a portfolio rebalancing:

* Identify the current portfolio allocation
* Compare it against the target allocation
* Calculate deviations for each asset class or sector
* Determine required buy/sell adjustments
* Check risk constraints (max exposure, volatility limits, liquidity)
* Propose rebalancing trades

